# Fase 6 — Geodatos de Panamá (Flujo D)

| Campo | Valor |
|---|---|
| **Rol líder** | Ingeniero de Datos |
| **Entrada** | API geoBoundaries (ADM2) + constantes INEC |
| **Salidas** | `data/processed/panama_distritos_merged.geojson` + `data/raw/inec_sociodemografico.csv` |
| **Doc de fase** | [`docs/analisis_proyecto/fases/fase6_geodatos.md`](../../docs/analisis_proyecto/fases/fase6_geodatos.md) |
| **Comando Make** | `make download-geodata` |

Construye el dataset geoespacial que alimenta el mapa coroplético del dashboard. La fuente real es geoBoundaries; los atributos sociodemográficos provienen de constantes derivadas del INEC con jitter determinístico a nivel de distrito.


## 0. Configuración

In [ ]:
import sys
from pathlib import Path

from src.paths import PROJECT_ROOT as ROOT, setup_path
setup_path()

import json
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.analisis_proyecto.geodata_panama import build_panama_geodata

RAW_GEOJSON = ROOT / 'data' / 'raw' / 'panama_distritos.geojson'
INEC_CSV = ROOT / 'data' / 'raw' / 'inec_sociodemografico.csv'
MERGED_GEOJSON = ROOT / 'data' / 'processed' / 'panama_distritos_merged.geojson'
MANIFEST = ROOT / 'data' / 'processed' / 'geodata_manifest.json'

sns.set_theme(style='whitegrid', context='notebook')
for d in (RAW_GEOJSON.parent, MERGED_GEOJSON.parent):
    d.mkdir(parents=True, exist_ok=True)
print(f'ROOT: {ROOT}')

## 1. Construir el geodataset (descarga + merge + jitter)

In [ ]:
report = build_panama_geodata(RAW_GEOJSON, INEC_CSV, MERGED_GEOJSON)
MANIFEST.write_text(json.dumps(report, indent=2), encoding='utf-8')
print(json.dumps(report, indent=2))

## 2. Inspeccionar la tabla sociodemográfica

In [ ]:
inec = pd.read_csv(INEC_CSV)
print(f'Distritos en INEC: {len(inec)}')
print(f'Provincias    : {inec["provincia"].nunique()}')
display(inec.head(10))
display(inec.describe(include='all').T)

## 3. Variables visualizadas en el mapa

In [ ]:
candidate_cols = [c for c in ['poblacion', 'superficie_agricola_ha',
                              'indice_pobreza', 'area_km2', 'exposure_risk'] if c in inec.columns]
print('Variables disponibles:', candidate_cols)
if 'provincia' in inec.columns and candidate_cols:
    by_prov = inec.groupby('provincia')[candidate_cols].mean(numeric_only=True).round(2)
    display(by_prov.sort_values(candidate_cols[0], ascending=False))

## 4. Visualización rápida del riesgo / variables principales

In [ ]:
if 'provincia' in inec.columns and candidate_cols:
    plot_col = candidate_cols[0]
    by_prov_sorted = (inec.groupby('provincia')[plot_col]
                          .mean()
                          .sort_values(ascending=True))
    fig, ax = plt.subplots(figsize=(9, 5))
    by_prov_sorted.plot(kind='barh', ax=ax, color='teal')
    ax.set_title(f'{plot_col} por provincia (media de distritos)')
    ax.set_xlabel(plot_col)
    plt.tight_layout()
    fig_path = ROOT / 'outputs' / 'chembl' / 'figures'
    fig_path.mkdir(parents=True, exist_ok=True)
    out = fig_path / 'panama_variable_by_province.png'
    fig.savefig(out, dpi=130)
    print(f'guardado: {out.relative_to(ROOT)}')
    plt.show()

## 5. Mapa coroplético interactivo (Plotly)

Renderizado embebido en el notebook. La misma lógica corre dentro de `viz/templates/analytics_map.html`.

In [ ]:
try:
    import plotly.express as px
    import geopandas as gpd

    gdf = gpd.read_file(MERGED_GEOJSON)
    geojson_dict = json.loads(gdf.to_json())
    var = candidate_cols[0] if candidate_cols else None
    if var is None:
        print('Sin variables sociodemográficas para colorear el mapa.')
    else:
        fig = px.choropleth(
            gdf,
            geojson=geojson_dict,
            locations='nombre_distrito',
            featureidkey='properties.nombre_distrito',
            color=var,
            color_continuous_scale='YlOrRd',
            scope='north america',
            hover_data=['provincia'] if 'provincia' in gdf.columns else None,
            title=f'Panamá — {var} por distrito',
        )
        fig.update_geos(fitbounds='locations', visible=False)
        fig.update_layout(height=520, margin=dict(l=0, r=0, t=40, b=0))
        fig.show()
except ImportError as exc:
    print(f'Plotly/GeoPandas no disponibles ({exc}). Salta la visualización interactiva.')

## 6. Validación: jitter determinístico

Re-ejecutar la construcción debe producir exactamente los mismos valores.

In [ ]:
import hashlib

csv_bytes = INEC_CSV.read_bytes()
before = hashlib.sha1(csv_bytes).hexdigest()
_ = build_panama_geodata(RAW_GEOJSON, INEC_CSV, MERGED_GEOJSON)
after = hashlib.sha1(INEC_CSV.read_bytes()).hexdigest()
print(f'SHA-1 antes : {before}')
print(f'SHA-1 después: {after}')
assert before == after, 'El pipeline no es determinístico — el jitter no usa seed por nombre'
print('Determinismo OK')

## 7. Limitaciones y disclaimer

- Las variables sociodemográficas (población, superficie agrícola, índice de pobreza) son **estimaciones derivadas del INEC**, no descargas en tiempo real. El campo `area_km2` sí proviene del GeoJSON real.
- El jitter por distrito es determinístico (`hash(nombre_distrito)`). Reproducible, pero no refleja variabilidad real intra-provincia.
- Datos oficiales del INEC a nivel distrito requerirán reemplazar la tabla generada por un CSV verificado.

## 8. Criterios de éxito

In [ ]:
checks = [
    ('GeoJSON merged generado', MERGED_GEOJSON.exists(), str(MERGED_GEOJSON.exists())),
    ('CSV INEC generado', INEC_CSV.exists(), str(INEC_CSV.exists())),
    ('>= 70 distritos', report['n_distritos'] >= 70, str(report['n_distritos'])),
    ('>= 10 provincias/comarcas', report['n_provincias'] >= 10, str(report['n_provincias'])),
    ('Pipeline determinístico', before == after, 'OK' if before == after else 'NO'),
    ('Manifest persistido', MANIFEST.exists(), str(MANIFEST.exists())),
]
df_check = pd.DataFrame(checks, columns=['Criterio', 'Pasa', 'Detalle'])
df_check['Pasa'] = df_check['Pasa'].map({True: 'OK', False: 'FALTA'})
df_check

---
*Fase anterior:* [`fase5_dashboard.ipynb`](fase5_dashboard.ipynb)  
*Siguiente fase:* [`fase7_comunicacion.ipynb`](fase7_comunicacion.ipynb)